# 30 — CNPJ por snapshot

Consulta os snapshots disponíveis do Cadastro Nacional da Pessoa Jurídica na Base dos Dados. Para cada snapshot é feita uma única consulta para todos os municípios selecionados; o resultado é então dividido localmente em lotes. Isso evita repetir a consulta ao BigQuery para cada lote.

A série deve ser interpretada como uma sequência de snapshots administrativos, não como painel anual de emprego.


In [ ]:
%pip -q install google-cloud-bigquery pandas pyarrow db-dtypes tqdm


In [ ]:
from pathlib import Path
import os, json, math
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "dados"
OUT_DIR = DATA_DIR / "processado"
CONTROL_DIR = DATA_DIR / "controle"
for p in [DATA_DIR, OUT_DIR, CONTROL_DIR]: p.mkdir(parents=True, exist_ok=True)

ARQUIVO_MUNICIPIOS = ROOT / "municipios.csv"
MUNICIPIOS_INLINE = [
    # "3516408",  # Franco da Rocha
    # "3525904",  # Jundiaí
]
LOTE_TAMANHO = 5

if ARQUIVO_MUNICIPIOS.exists():
    mun = pd.read_csv(ARQUIVO_MUNICIPIOS, dtype=str)
    if "id_municipio" not in mun.columns:
        raise ValueError("municipios.csv deve conter a coluna id_municipio")
    ids = mun["id_municipio"].astype(str).str.strip().dropna().tolist()
else:
    ids = [str(x).strip() for x in MUNICIPIOS_INLINE if str(x).strip()]

ids = list(dict.fromkeys(ids))
if not ids:
    raise ValueError("Informe municípios em municipios.csv ou em MUNICIPIOS_INLINE.")
if any(len(x) != 7 or not x.isdigit() for x in ids):
    raise ValueError("Todos os códigos devem ser códigos IBGE municipais de 7 dígitos.")

LOTES = [ids[i:i+LOTE_TAMANHO] for i in range(0, len(ids), LOTE_TAMANHO)]
print(f"Municípios: {len(ids)} | lotes: {len(LOTES)} | tamanho máximo: {LOTE_TAMANHO}")

from google.cloud import bigquery
BILLING_PROJECT_ID=os.getenv("BIGQUERY_PROJECT")
if not BILLING_PROJECT_ID:
    raise EnvironmentError("Defina BIGQUERY_PROJECT com um projeto Google Cloud habilitado para cobrança do BigQuery.")
bq=bigquery.Client(project=BILLING_PROJECT_ID)


In [ ]:
snap=bq.query("""SELECT CAST(data AS STRING) snapshot FROM `basedosdados.br_me_cnpj.estabelecimentos` WHERE data IS NOT NULL GROUP BY data ORDER BY data""").to_dataframe()
SNAPSHOTS=snap.snapshot.astype(str).str[:10].tolist()
print("Snapshots:",len(SNAPSHOTS),"|",SNAPSHOTS[0],"a",SNAPSHOTS[-1])


In [ ]:
def sql_snapshot(s):
    ids_sql=','.join(f"'{x}'" for x in ids)
    return f"""
WITH e AS (
 SELECT * FROM `basedosdados.br_me_cnpj.estabelecimentos`
 WHERE data=DATE('{s}') AND id_municipio IN ({ids_sql})
), b AS (SELECT DISTINCT cnpj_basico FROM e),
p AS (
 SELECT p.* FROM `basedosdados.br_me_cnpj.empresas` p
 INNER JOIN b USING(cnpj_basico) WHERE p.data=DATE('{s}')
), sm AS (
 SELECT sm.* FROM `basedosdados.br_me_cnpj.simples` sm
 INNER JOIN b USING(cnpj_basico)
)
SELECT e.*, p.razao_social, p.natureza_juridica, p.qualificacao_responsavel,
       p.ente_federativo, p.capital_social, p.porte,
       sm.opcao_simples, sm.data_opcao_simples, sm.data_exclusao_simples,
       sm.opcao_mei, sm.data_opcao_mei, sm.data_exclusao_mei
FROM e
LEFT JOIN p USING(cnpj_basico)
LEFT JOIN sm USING(cnpj_basico)
"""


In [ ]:
from tqdm.auto import tqdm
base=OUT_DIR/"cnpj"; base.mkdir(parents=True,exist_ok=True)
for s in tqdm(SNAPSHOTS, desc="CNPJ snapshots"):
    outs=[base/f"cnpj_lote{i:02d}_{s}.parquet" for i in range(1,len(LOTES)+1)]
    if all(p.exists() for p in outs): continue
    df=bq.query(sql_snapshot(s)).to_dataframe(create_bqstorage_client=False)
    if df.empty: raise RuntimeError(f"Snapshot {s} retornou zero linhas")
    df["id_municipio"]=df["id_municipio"].astype(str)
    for i,lote in enumerate(LOTES,1):
        out=outs[i-1]
        if out.exists(): continue
        dg=df[df.id_municipio.isin(lote)].copy()
        if dg.empty: raise RuntimeError(f"Snapshot {s}, lote {i}: zero linhas")
        dg.to_parquet(out,index=False,compression="snappy")
print("CNPJ concluído em",base)
